# Analisis Exploratorio de Datos (EDA) — Gasto Publico MEF

**Fase 5 del pipeline v2**

Antes de construir modelos de ML necesitamos entender los datos que tenemos.
Este notebook responde preguntas concretas:

- Como se distribuye el gasto? Hay pocos departamentos que concentran la mayoria?
- Que sectores reciben mas presupuesto?
- Existen outliers extremos que podrian distorsionar un modelo?
- Que variables tienen correlacion y cuales son independientes?
- Que features tienen sentido para un modelo predictivo?

Las conclusiones de este analisis guian directamente las decisiones de la Fase 6 (ML).

## 1. Setup y carga de datos

In [ ]:
import os
import sys

# Agregar la raiz del proyecto al path para importar config/ y src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from config.logging_config import setup_logging
from config.settings import settings
from config.spark_config import crear_spark_session

setup_logging()

# Estilo de graficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print(f"Raiz del proyecto: {ROOT}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

In [ ]:
# Iniciar Spark y cargar los datos
spark = crear_spark_session(app_name="eda_fase5")

from src.calidad.filtros_uat import aplicar_filtros_uat
from src.ingesta.lector_csv import leer_csv_con_schema

# Si ya descargaste el CSV completo, apunta a el.
# Si no, usa el sample de tests para explorar la estructura.
RUTA_CSV = settings.data.ruta_csv_raw
RUTA_SAMPLE = os.path.join(ROOT, "tests", "data", "sample_gasto.csv")

ruta = str(RUTA_CSV) if RUTA_CSV.exists() else RUTA_SAMPLE
print(f"Usando: {ruta}")

df_spark = leer_csv_con_schema(spark, ruta)
df_limpio, metricas_calidad = aplicar_filtros_uat(df_spark, settings.calidad)

print(f"\nMetricas de calidad: {metricas_calidad}")

In [ ]:
# Convertir a pandas para visualizacion
# (Spark es para procesamiento masivo; pandas + matplotlib para graficos)
df = df_limpio.toPandas()

print(f"Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head(10)

## 2. Estadisticas descriptivas

Primer vistazo a los numeros: como se ve el gasto "tipico" y que tan dispersos estan los valores.

In [ ]:
# Info general del DataFrame
df.info()

In [ ]:
# Estadisticas del monto devengado
monto = df["MONTO_DEVENGADO"]

estadisticas = pd.DataFrame({
    "Metrica": [
        "Registros", "Media", "Mediana", "Desviacion estandar",
        "Minimo", "Percentil 25", "Percentil 50", "Percentil 75",
        "Percentil 95", "Percentil 99", "Maximo",
        "Asimetria (skewness)", "Curtosis",
    ],
    "Valor": [
        f"{len(monto):,}",
        f"{monto.mean():,.2f}",
        f"{monto.median():,.2f}",
        f"{monto.std():,.2f}",
        f"{monto.min():,.2f}",
        f"{monto.quantile(0.25):,.2f}",
        f"{monto.quantile(0.50):,.2f}",
        f"{monto.quantile(0.75):,.2f}",
        f"{monto.quantile(0.95):,.2f}",
        f"{monto.quantile(0.99):,.2f}",
        f"{monto.max():,.2f}",
        f"{monto.skew():.2f}",
        f"{monto.kurtosis():.2f}",
    ],
})

print("Estadisticas de MONTO_DEVENGADO")
print("=" * 45)
estadisticas

## 3. Distribucion del monto devengado

Dos preguntas: cual es la forma de la distribucion, y necesitamos transformarla (log) para un modelo?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma original
axes[0].hist(monto, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].axvline(monto.mean(), color="red", linestyle="--", label=f"Media: {monto.mean():,.0f}")
axes[0].axvline(monto.median(), color="orange", linestyle="--", label=f"Mediana: {monto.median():,.0f}")
axes[0].set_title("Distribucion del monto devengado")
axes[0].set_xlabel("Monto (S/)")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Histograma en escala log (si hay rango amplio)
monto_positivo = monto[monto > 0]
if monto_positivo.max() / monto_positivo.min() > 100:
    axes[1].hist(np.log10(monto_positivo), bins=50, edgecolor="black", alpha=0.7, color="seagreen")
    axes[1].set_title("Distribucion log10(monto)")
    axes[1].set_xlabel("log10(Monto)")
    axes[1].set_ylabel("Frecuencia")
else:
    axes[1].hist(monto, bins=50, edgecolor="black", alpha=0.7, color="seagreen")
    axes[1].set_title("Distribucion del monto (rango estrecho, no necesita log)")

# Boxplot
axes[2].boxplot(monto, vert=True, patch_artist=True,
                boxprops=dict(facecolor="lightyellow"))
axes[2].set_title("Boxplot del monto devengado")
axes[2].set_ylabel("Monto (S/)")

plt.tight_layout()
plt.show()

## 4. Gasto por departamento

Hay concentracion? Pocos departamentos se llevan la mayor parte del presupuesto?

In [ ]:
# Tabla resumen por departamento
if "DEPARTAMENTO_EJECUTORA_NOMBRE" in df.columns:
    col_depto = "DEPARTAMENTO_EJECUTORA_NOMBRE"
else:
    col_depto = "DISTRITO_EJECUTORA_NOMBRE"  # fallback para sample

por_depto = (
    df.groupby(col_depto)["MONTO_DEVENGADO"]
    .agg(["sum", "mean", "median", "count", "std"])
    .sort_values("sum", ascending=False)
    .rename(columns={
        "sum": "Total", "mean": "Promedio", "median": "Mediana",
        "count": "Registros", "std": "Desv. Est."
    })
)

# Porcentaje acumulado para ver concentracion
por_depto["% del Total"] = (por_depto["Total"] / por_depto["Total"].sum() * 100).round(1)
por_depto["% Acumulado"] = por_depto["% del Total"].cumsum().round(1)

print(f"Departamentos/distritos unicos: {len(por_depto)}")
por_depto.head(15)

In [ ]:
# Top 15 por gasto total
top_n = min(15, len(por_depto))
top = por_depto.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras horizontales — gasto total
axes[0].barh(range(top_n), top["Total"], color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel("Gasto total (S/)")
axes[0].set_title(f"Top {top_n} por gasto total")

# Barras — cantidad de registros
axes[1].barh(range(top_n), top["Registros"], color="coral", edgecolor="black", alpha=0.8)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top.index, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel("Cantidad de registros")
axes[1].set_title(f"Top {top_n} por volumen de registros")

plt.tight_layout()
plt.show()

# Curva de concentracion (Pareto)
if len(por_depto) > 5:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(1, len(por_depto) + 1)
    ax.plot(x, por_depto["% Acumulado"].values, marker="o", markersize=4, color="steelblue")
    ax.axhline(80, color="red", linestyle="--", alpha=0.7, label="80% del gasto")
    ax.set_xlabel("Departamentos (ordenados por gasto)")
    ax.set_ylabel("% acumulado del gasto")
    ax.set_title("Curva de concentracion del gasto (tipo Pareto)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Gasto por sector

Que areas del gobierno reciben mas presupuesto? Educacion, Salud, Defensa?

In [ ]:
por_sector = (
    df.groupby("SECTOR_NOMBRE")["MONTO_DEVENGADO"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
    .rename(columns={"sum": "Total", "mean": "Promedio", "count": "Registros"})
)
por_sector["% del Total"] = (por_sector["Total"] / por_sector["Total"].sum() * 100).round(1)

print(f"Sectores unicos: {len(por_sector)}")
por_sector.head(15)

In [ ]:
# Grafico de barras por sector
top_sectores = min(15, len(por_sector))
top_s = por_sector.head(top_sectores)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(top_sectores), top_s["Total"], color="teal", edgecolor="black", alpha=0.8)
ax.set_yticks(range(top_sectores))
ax.set_yticklabels(top_s.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Gasto total (S/)")
ax.set_title(f"Top {top_sectores} sectores por gasto total")
plt.tight_layout()
plt.show()

## 6. Gasto por mes

Hay estacionalidad? El gasto se concentra en ciertos meses (tipicamente fin de ano fiscal)?

In [ ]:
por_mes = (
    df.groupby("MES_EJE")["MONTO_DEVENGADO"]
    .agg(["sum", "mean", "count"])
    .sort_index()
    .rename(columns={"sum": "Total", "mean": "Promedio", "count": "Registros"})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(por_mes.index, por_mes["Total"], color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_xlabel("Mes")
axes[0].set_ylabel("Gasto total (S/)")
axes[0].set_title("Gasto total por mes")

axes[1].bar(por_mes.index, por_mes["Registros"], color="coral", edgecolor="black", alpha=0.8)
axes[1].set_xlabel("Mes")
axes[1].set_ylabel("Cantidad de registros")
axes[1].set_title("Volumen de registros por mes")

plt.tight_layout()
plt.show()

por_mes

## 7. Analisis de correlacion

Que variables numericas estan relacionadas entre si? Esto nos dice que features
aportan informacion independiente al modelo y cuales son redundantes.

In [ ]:
# Seleccionar columnas numericas
numericas = df.select_dtypes(include=[np.number])

if numericas.shape[1] >= 2:
    corr = numericas.corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
        center=0, square=True, linewidths=0.5, ax=ax,
        vmin=-1, vmax=1,
    )
    ax.set_title("Matriz de correlacion (variables numericas)")
    plt.tight_layout()
    plt.show()
else:
    print("Solo hay una columna numerica (MONTO_DEVENGADO).")
    print("Para correlacion necesitamos las columnas de montos adicionales")
    print("del CSV completo (MONTO_PIA, MONTO_PIM, MONTO_CERTIFICADO, etc.).")
    print("\nCon el dataset completo, el heatmap mostrara como se relacionan")
    print("las diferentes etapas del gasto presupuestal.")

## 8. Deteccion de outliers

Dos metodos complementarios:
- **IQR (Rango Intercuartilico)**: robusto, no asume distribucion normal
- **Z-score**: asume normalidad, mide desviaciones estandar desde la media

In [ ]:
# Metodo 1: IQR
Q1 = monto.quantile(0.25)
Q3 = monto.quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_iqr = monto[(monto < limite_inferior) | (monto > limite_superior)]

print("Deteccion de outliers por IQR")
print("=" * 40)
print(f"Q1 (percentil 25): {Q1:,.2f}")
print(f"Q3 (percentil 75): {Q3:,.2f}")
print(f"IQR: {IQR:,.2f}")
print(f"Limite inferior: {limite_inferior:,.2f}")
print(f"Limite superior: {limite_superior:,.2f}")
print(f"\nOutliers detectados: {len(outliers_iqr):,} ({len(outliers_iqr)/len(monto)*100:.1f}%)")

In [ ]:
# Metodo 2: Z-score
z_scores = np.abs(stats.zscore(monto))
umbral_z = 3  # mas de 3 desviaciones estandar
outliers_z = monto[z_scores > umbral_z]

print("Deteccion de outliers por Z-score")
print("=" * 40)
print(f"Umbral: |z| > {umbral_z}")
print(f"Outliers detectados: {len(outliers_z):,} ({len(outliers_z)/len(monto)*100:.1f}%)")

# Visualizacion comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IQR
axes[0].hist(monto, bins=50, edgecolor="black", alpha=0.6, color="steelblue", label="Normal")
if len(outliers_iqr) > 0:
    axes[0].hist(outliers_iqr, bins=30, edgecolor="black", alpha=0.7, color="red", label="Outliers IQR")
axes[0].axvline(limite_superior, color="red", linestyle="--", alpha=0.8)
axes[0].set_title(f"Outliers por IQR ({len(outliers_iqr):,} detectados)")
axes[0].set_xlabel("Monto (S/)")
axes[0].legend()

# Z-score
axes[1].hist(z_scores, bins=50, edgecolor="black", alpha=0.7, color="seagreen")
axes[1].axvline(umbral_z, color="red", linestyle="--", label=f"Umbral z={umbral_z}")
axes[1].set_title(f"Distribucion de Z-scores ({len(outliers_z):,} outliers)")
axes[1].set_xlabel("|Z-score|")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Distribucion por nivel de gobierno

Gobierno Nacional vs Regional vs Local — como se reparte el presupuesto?

In [ ]:
if "NIVEL_GOBIERNO_NOMBRE" in df.columns:
    por_nivel = (
        df.groupby("NIVEL_GOBIERNO_NOMBRE")["MONTO_DEVENGADO"]
        .agg(["sum", "mean", "median", "count"])
        .sort_values("sum", ascending=False)
        .rename(columns={"sum": "Total", "mean": "Promedio", "median": "Mediana", "count": "Registros"})
    )
    por_nivel["% del Total"] = (por_nivel["Total"] / por_nivel["Total"].sum() * 100).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart
    axes[0].pie(
        por_nivel["Total"], labels=por_nivel.index, autopct="%1.1f%%",
        colors=["steelblue", "coral", "seagreen"], startangle=90
    )
    axes[0].set_title("Distribucion del gasto por nivel de gobierno")

    # Boxplot comparativo
    niveles = df["NIVEL_GOBIERNO_NOMBRE"].unique()
    datos_box = [df[df["NIVEL_GOBIERNO_NOMBRE"] == n]["MONTO_DEVENGADO"] for n in niveles]
    axes[1].boxplot(datos_box, labels=niveles, patch_artist=True,
                    boxprops=dict(facecolor="lightyellow"))
    axes[1].set_ylabel("Monto devengado (S/)")
    axes[1].set_title("Dispersion del monto por nivel de gobierno")
    axes[1].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.show()

    por_nivel
else:
    print("Columna NIVEL_GOBIERNO_NOMBRE no disponible en este dataset.")

## 10. Conclusiones para Feature Engineering (Fase 6)

Basandonos en lo observado, estas son las decisiones para los modelos de ML:

In [ ]:
# Resumen cuantitativo del analisis
resumen = {
    "Total registros limpios": f"{len(df):,}",
    "Rango de montos": f"{monto.min():,.2f} — {monto.max():,.2f}",
    "Media vs Mediana": f"{monto.mean():,.2f} vs {monto.median():,.2f}",
    "Asimetria (skewness)": f"{monto.skew():.2f}",
    "Outliers IQR": f"{len(outliers_iqr):,} ({len(outliers_iqr)/len(monto)*100:.1f}%)",
    "Outliers Z-score": f"{len(outliers_z):,} ({len(outliers_z)/len(monto)*100:.1f}%)",
}

if "DEPARTAMENTO_EJECUTORA_NOMBRE" in df.columns:
    resumen["Departamentos unicos"] = str(df["DEPARTAMENTO_EJECUTORA_NOMBRE"].nunique())
resumen["Sectores unicos"] = str(df["SECTOR_NOMBRE"].nunique())

print("RESUMEN DEL EDA")
print("=" * 50)
for k, v in resumen.items():
    print(f"  {k:.<35s} {v}")

### Decisiones para la Fase 6 (ML)

**Features candidatas para el modelo de regresion:**
- `DEPARTAMENTO_EJECUTORA_NOMBRE` — alta variabilidad entre departamentos
- `SECTOR_NOMBRE` — diferencias claras de presupuesto por sector
- `NIVEL_GOBIERNO_NOMBRE` — Nacional vs Regional vs Local
- `MES_EJE` — posible estacionalidad

**Transformaciones necesarias:**
- Encoding categorico: OneHotEncoder o LabelEncoder para las variables string
- Si la distribucion es muy asimetrica (skewness > 2): considerar log-transform del target
- Estandarizacion (StandardScaler) de features numericas

**Sobre outliers:**
- No eliminar outliers a ciegas — en gasto publico, los montos altos son reales
- Usar modelos robustos a outliers (Random Forest, Gradient Boosting) en vez de regresion lineal pura
- Evaluar si un modelo separado para montos extremos mejora la prediccion

**Para clustering:**
- Agrupar por (departamento, sector) y usar las metricas agregadas como features
- Features: total_monto, promedio_monto, num_registros, std_monto
- Normalizar antes de K-Means (sensible a escala)

In [ ]:
# Cerrar SparkSession
spark.stop()
print("SparkSession cerrada. EDA completado.")